# ดึงแปลงที่ดินจาก LandsMaps ตามไฟล์ขอบเขต

ทางเลือกสำหรับคนที่ชอบ Jupyter ได้ผลเหมือน `python fetch_parcels.py` ทุกอย่าง

1. วางไฟล์ขอบเขต (KMZ, KML, SHP, GPKG, GeoJSON, โฟลเดอร์ .gdb) ไว้ใน `areas/`
2. แก้ `AREA_FILE` ด้านล่าง (หรือปล่อยเป็น `None` = ทุกไฟล์ใน `areas/`)
3. Run All

ผลลัพธ์อยู่ที่ `output/<ชื่อไฟล์>/` รายละเอียดดูที่ README.md

In [ ]:
import sys
from pathlib import Path

# หาโฟลเดอร์โปรเจกต์ (ที่มี landsmaps/) ไม่ว่า kernel จะเริ่มจากโฟลเดอร์ไหน
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "landsmaps").is_dir())
sys.path.insert(0, str(ROOT))

from landsmaps import run_area
from landsmaps.area import find_area_files
from landsmaps.config import AREAS_DIR

# ─────────────────────────── ตั้งค่า ───────────────────────────
AREA_FILE = None            # เช่น ROOT / "areas/my_site.kmz"  (None = ทุกไฟล์ใน areas/)
FETCH_DETAIL = True         # False = เอาแค่รูปแปลง ไม่ใช้โควตา API รายละเอียด

In [ ]:
files = [Path(AREA_FILE)] if AREA_FILE else find_area_files(AREAS_DIR)
print("พื้นที่:", [f.name for f in files] or f"ไม่พบไฟล์ใน {AREAS_DIR}")

results = []
for f in files:
    r = run_area(f, fetch_detail=FETCH_DETAIL)
    results.append(r)
    FETCH_DETAIL = FETCH_DETAIL and not r.quota_hit     # โควตาเต็มแล้ว ไม่ต้องลองพื้นที่ถัดไป
    print()

In [ ]:
import geopandas as gpd

# ดูตัวอย่างผลลัพธ์ของพื้นที่แรก
gpd.read_file(results[0].parcel_file).drop(columns="geometry").head(10) if results else None